# Analytics Intelligence Platform — Exploratory Data Analysis

This notebook explores all four raw data sources ingested into the `web_analytics` PostgreSQL database.
Each section connects to the live DB via `utils/db.py` and produces charts using matplotlib/plotly.

**Sections:**
1. Data Overview
2. Traffic Analysis
3. User Behavior
4. Conversion Analysis
5. SEO Analysis
6. Anomaly Detection
7. Key Findings

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

from utils.db import query_df

# Output directory for saved plots
PLOT_DIR = Path('..') / 'data' / 'processed' / 'eda_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print('Environment ready. Plot directory:', PLOT_DIR.resolve())

---
## Section 1 — Data Overview

Connect to PostgreSQL and inspect each raw table: row counts, date ranges, and column names.

In [ ]:
# Row counts for all raw tables
RAW_TABLES = [
    'raw_ga4_sessions',
    'raw_server_logs',
    'raw_clickstream_events',
    'raw_scrape_pages',
]

print('=' * 55)
print(f'{'Table':<30} {'Rows':>10}')
print('=' * 55)
for tbl in RAW_TABLES:
    cnt = query_df(f'SELECT COUNT(*) n FROM {tbl}')
    print(f'{tbl:<30} {int(cnt["n"].iloc[0]):>10,}')
print('=' * 55)

In [ ]:
# Date ranges for each table
DATE_COLS = {
    'raw_ga4_sessions':       'session_date',
    'raw_server_logs':        'log_time',
    'raw_clickstream_events': 'event_time',
    'raw_scrape_pages':       'scraped_at',
}

print(f'{'Table':<30} {'Min date':<22} {'Max date':<22}')
print('-' * 75)
for tbl, col in DATE_COLS.items():
    df = query_df(f'SELECT MIN({col})::DATE mn, MAX({col})::DATE mx FROM {tbl}')
    print(f'{tbl:<30} {str(df["mn"].iloc[0]):<22} {str(df["mx"].iloc[0]):<22}')

In [ ]:
# Column names for each table
for tbl in RAW_TABLES:
    df = query_df(f'SELECT * FROM {tbl} LIMIT 0')
    print(f'\n{tbl} ({len(df.columns)} columns):')
    print('  ' + ', '.join(df.columns.tolist()))